# Train your Nullified AI model (free)

Runs on Colab's **free T4 GPU**. In the menu: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

You need a Hugging Face token with **Write** access (https://huggingface.co/settings/tokens). Paste it when asked.

Trains a **7B coder** (Qwen2.5-Coder, Apache-2.0) on the real Fabric 1.21.x mods the collector gathered (the `repos3` batch). A 7B fits the free T4 comfortably and trains in ~15–30 min. (gpt-oss-20B needs a bigger GPU — see `training/nullified_train_kaggle.ipynb` and the README.) It saves progress, so a disconnect isn't fatal — just Run all again.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi -L || print('No GPU! Runtime -> Change runtime type -> T4 GPU, then Run all again.')

In [ ]:
#@title 2. Install (a few minutes)
!pip install -q unsloth 'huggingface_hub>=0.26' >/dev/null 2>&1
!git clone -q https://github.com/surgeclient/nullified-ai.git /content/nullified-ai || (cd /content/nullified-ai && git pull -q)
print('ready')

In [ ]:
#@title 3. Your Hugging Face token (Write access)
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face WRITE token (hf_...): ').strip()
from huggingface_hub import HfApi
print('signed in as', HfApi(token=os.environ['HF_TOKEN']).whoami()['name'])

In [ ]:
#@title 4. Settings
BASE = 'unsloth/Qwen2.5-Coder-7B-Instruct'  #@param {type:'string'}
RUNS = 'repos3'  #@param {type:'string'}
ADAPTER_REPO = 'nullified-coder'  #@param {type:'string'}
EPOCHS = 3.0  #@param {type:'number'}
MAX_SEQ = 4096  #@param {type:'integer'}  # T4 has no flash-attention; 8192 OOMs on long mods. Raise on a bigger GPU.
RANK = 32  #@param {type:'integer'}
print('base:', BASE, '| runs:', RUNS, '| max_seq:', MAX_SEQ)

In [ ]:
#@title 5. Train + upload (the long step)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!cd /content/nullified-ai && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python training/train_lora.py --base "$BASE" --runs "$RUNS" --adapter-repo "$ADAPTER_REPO" --epochs $EPOCHS --max-seq-train $MAX_SEQ --rank $RANK --gguf


When it finishes it prints your model repo (ending `-gguf`). Use it on the website's Build tab under **Options → Model** (or `--model-repo <you>/nullified-coder-gguf` with the command line).